# Stage 2. Control image quality

## 1. Setup

### 1.1 Imports

In [ ]:
import datetime
import logging
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tifffile
from omegaconf import OmegaConf
from scipy.ndimage import median_filter

from acid.config import load_config, prepare_output
from acid.image_measurement.measure_glob_image_stat import (
    axis_image_stat,
    normalized_intensity_skewness,
)
from acid.image_quality_control.display_qc import plot_qc_by_channel_condition
from acid.image_quality_control.flag_column import add_flag_column
from acid.image_quality_control.measure_mean_over_std import mean_over_std
from acid.image_quality_control.measure_plls import compute_plls
from acid.utils.fov_axis_utils import get_fov_ch_shape
from acid.utils.metadata.filtering import filter_metadata_by_splits
from acid.utils.metadata.loading import load_metadata

### 1.2 Configure logging

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

### 1.3 Load configuration

In [ ]:
CONFIG_FILE = None  # "configs/exp_001.yaml"
config = load_config(CONFIG_FILE)
prepare_output(config)
stage_cfg = config.quality_control
paths_cfg = config.shared.paths

## 2. Input Data Preparation

In [ ]:
metadata_df, metadata_file_name = load_metadata(stage_cfg.metadata)
metadata_df = filter_metadata_by_splits(
    metadata_df, selected_splits="train", split_config=stage_cfg.dataset_split
).copy()
if metadata_df.empty:
    raise ValueError("No training images selected")

## 3. Quality control
### 3.1 Helpers

In [ ]:
# FUTURE: move to src/acid/image_quality_control/measure_quality.py
def measure_quality(metadata_df, cfg):
    metadata_df = metadata_df.copy()
    _fov_shape, num_channels, _shape_of_channels = get_fov_ch_shape(
        metadata_df,
        cfg.processing.fov_directory,
        fov_clm=cfg.metadata.dataframe_columns.fov_column_name,
        channel_axis=cfg.processing.channel_axis,
        null_value=cfg.metadata.dataframe_columns.null_value,
    )
    glob_plls_collection_list = []
    glob_skewness_collection_list = []
    glob_mean_over_std_collection_list = []
    for i in metadata_df.index:
        print("---------    ---------")
        field_of_view_file = metadata_df.loc[
            i, cfg.metadata.dataframe_columns.fov_column_name
        ]
        print(f"working on {field_of_view_file}")
        try:
            field_of_view = tifffile.imread(
                os.path.join(cfg.processing.fov_directory, str(field_of_view_file))
            )
        except Exception as error:
            logging.getLogger(__name__).warning(
                "QC calculation failed: %s", error, exc_info=True
            )
            print(f"can't open {field_of_view_file}, skipping QC calculations")
            glob_plls_collection_list.append(
                [
                    cfg.metadata.dataframe_columns.null_value
                    for qc in range(num_channels)
                ]
            )
            glob_skewness_collection_list.append(
                [
                    cfg.metadata.dataframe_columns.null_value
                    for qc in range(num_channels)
                ]
            )
            glob_mean_over_std_collection_list.append(
                [
                    cfg.metadata.dataframe_columns.null_value
                    for qc in range(num_channels)
                ]
            )
            continue
        try:
            fov_plls = compute_plls(
                image=field_of_view, axis=cfg.processing.channel_axis
            )
            glob_plls_collection_list.append(fov_plls)
            print("plls calculated")
        except Exception as error:
            logging.getLogger(__name__).warning(
                "QC calculation failed: %s", error, exc_info=True
            )
            print(f"can't calculate plls for {field_of_view_file}")
            glob_plls_collection_list.append(
                [
                    cfg.metadata.dataframe_columns.null_value
                    for qc in range(num_channels)
                ]
            )
        try:
            if cfg.processing.median_smooth:
                median_kwargs = {
                    cfg.processing.median_kwargs_axes_name: tuple(
                        [
                            c
                            for c in range(len(field_of_view.shape))
                            if c != cfg.processing.channel_axis
                        ]
                    )
                }
                smoothed_field_of_view = median_filter(
                    field_of_view, size=cfg.processing.median_size, **median_kwargs
                )
            else:
                smoothed_field_of_view = field_of_view
            print(smoothed_field_of_view.shape)
            fov_skewness = axis_image_stat(
                image=smoothed_field_of_view,
                stat=normalized_intensity_skewness,
                axis=cfg.processing.channel_axis,
                mask=cfg.processing.mask,
                normalize=tuple(cfg.processing.normalize),
                ravel_kwargs=OmegaConf.to_container(
                    cfg.processing.ravel_kwargs, resolve=True
                ),
            )
            glob_skewness_collection_list.append(fov_skewness)
            print("skewness calculated")
        except Exception as error:
            logging.getLogger(__name__).warning(
                "QC calculation failed: %s", error, exc_info=True
            )
            print(f"can't calculate skweness for {field_of_view_file}")
            glob_skewness_collection_list.append(
                [
                    cfg.metadata.dataframe_columns.null_value
                    for qc in range(num_channels)
                ]
            )
        try:
            fov_mean_over_std = mean_over_std(
                image=field_of_view,
                axis=cfg.processing.channel_axis,
                null_val=cfg.metadata.dataframe_columns.null_value,
                inf_val=cfg.metadata.dataframe_columns.null_value,
            )
            glob_mean_over_std_collection_list.append(fov_mean_over_std)
            print("mean over std calculated")
        except Exception as error:
            logging.getLogger(__name__).warning(
                "QC calculation failed: %s", error, exc_info=True
            )
            print(f"can't calculate mean over std for {field_of_view_file}")
            glob_mean_over_std_collection_list.append(
                [
                    cfg.metadata.dataframe_columns.null_value
                    for qc in range(num_channels)
                ]
            )
    glob_plls = np.stack(glob_plls_collection_list, axis=0)
    glob_skewnwss = np.stack(glob_skewness_collection_list, axis=0)
    glob_mean_over_std = np.stack(glob_mean_over_std_collection_list, axis=0)
    plls_column_names = [
        f"{cfg.metadata.dataframe_columns.plls_column_name}{cfg.metadata.dataframe_columns.channel_measurement_separator}{ch}"
        for ch in range(num_channels)
    ]
    skewness_column_names = [
        f"{cfg.metadata.dataframe_columns.skewness_column_name}{cfg.metadata.dataframe_columns.channel_measurement_separator}{ch}"
        for ch in range(num_channels)
    ]
    mean_over_std_column_names = [
        f"{cfg.metadata.dataframe_columns.mean_over_std_column_name}{cfg.metadata.dataframe_columns.channel_measurement_separator}{ch}"
        for ch in range(num_channels)
    ]
    metadata_df[plls_column_names] = pd.DataFrame(glob_plls, index=metadata_df.index)
    metadata_df[skewness_column_names] = pd.DataFrame(
        glob_skewnwss, index=metadata_df.index
    )
    metadata_df[mean_over_std_column_names] = pd.DataFrame(
        glob_mean_over_std, index=metadata_df.index
    )
    print("finished")
    return metadata_df, num_channels, mean_over_std_column_names, skewness_column_names


# FUTURE: move to src/acid/image_quality_control/flag_column.py
def flag_quality(metadata_df, columns, cfg):
    upper = list(cfg.processing.mean_over_std_lowpass_thresholds)
    lower = list(cfg.processing.mean_over_std_highpass_thresholds)
    if len(upper) != len(columns) or len(lower) != len(columns):
        raise ValueError("QC threshold counts must match the number of channels")
    return add_flag_column(
        df=metadata_df,
        lowpass_thres=upper,
        highpass_thres=lower,
        cols=columns,
        flag_col=cfg.metadata.dataframe_columns.flag_column_name,
        flag_value=cfg.processing.flag_value,
        ok_value=cfg.processing.ok_value,
    )

### 3.2 Measure and flag training images

In [ ]:
metadata_df, num_channels, mean_over_std_column_names, skewness_column_names = (
    measure_quality(metadata_df, stage_cfg)
)
flagged_metadata_df = flag_quality(metadata_df, mean_over_std_column_names, stage_cfg)

## 4. Save metadata

In [ ]:
# FUTURE: move to src/acid/utils/metadata/saving.py
def save_metadata_dataframe(metadata_df, metadata_config, project_name):
    saving = metadata_config.saving
    separator = saving.save_file_name_separator
    suffix = saving.metadata_file_suffix.format(save_file_name_separator=separator)
    filename = separator.join(
        [
            datetime.datetime.now().astimezone().strftime(saving.metadata_date_format),
            project_name,
            saving.metadata_savingword,
            suffix,
        ]
    )
    path = Path(metadata_config.directory) / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    metadata_df.to_csv(path, index=saving.save_csv_index)
    logging.getLogger(__name__).info("Saved metadata: %s", path)
    return path


metadata_path = save_metadata_dataframe(
    flagged_metadata_df, stage_cfg.metadata, config.project_identity.project_name
)

## 5. Inspect results

In [ ]:
flagged_metadata_df.head()

### 5.1 QC plots

In [ ]:
# FUTURE: move to src/acid/image_quality_control/display_qc.py
def plot_quality(
    metadata_df, mean_columns, skew_columns, num_channels, cfg, project_name
):
    graphs = cfg.graphs_saving
    output = Path(graphs.graph_saving_directory)
    if graphs.save_graphs:
        output.mkdir(parents=True, exist_ok=True)
    figures = []
    for columns, title, word in (
        (
            mean_columns,
            "Mean / standard deviation",
            graphs.mean_over_std_per_channel_savingword,
        ),
        (
            skew_columns,
            "Normalized intensity skewness",
            graphs.skewness_per_channel_savingword,
        ),
    ):
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.violinplot(data=metadata_df[columns], ax=ax, inner=None)
        sns.stripplot(data=metadata_df[columns], ax=ax, color="black", size=3)
        ax.set_title(title)
        ax.tick_params(axis="x", labelrotation=90)
        figures.append((fig, word))
    columns = cfg.metadata.dataframe_columns
    for condition, word in (
        (columns.experiment_column_name, graphs.mean_over_std_per_ch_exp_savingword),
        (columns.well_column_name, graphs.mean_over_std_per_ch_well_savingword),
    ):
        fig, axes = plt.subplots(
            num_channels, 1, figsize=(10, 5 * num_channels), squeeze=False
        )
        plot_qc_by_channel_condition(
            metadata_df=metadata_df,
            qc_clm=columns.mean_over_std_column_name,
            condition_clm=condition,
            ax=axes[:, 0],
            num_channels=num_channels,
            ch_measurement_separator=columns.channel_measurement_separator,
        )
        figures.append((fig, word))
    for fig, word in figures:
        if graphs.save_graphs:
            filename = f"{datetime.datetime.now().astimezone().strftime(graphs.graph_date_format)}_{project_name}_{word}{graphs.graph_suffix}"
            fig.savefig(output / filename, bbox_inches="tight")
    plt.show()


plot_quality(
    flagged_metadata_df,
    mean_over_std_column_names,
    skewness_column_names,
    num_channels,
    stage_cfg,
    config.project_identity.project_name,
)

### 5.2 Inspect values near a target

In [ ]:
display_cfg = stage_cfg.visualization
for selection in (display_cfg.mean_over_std, display_cfg.skewness):
    column = f"{selection.metric}{stage_cfg.metadata.dataframe_columns.channel_measurement_separator}{selection.channel}"
    values = metadata_df[column]
    selected = metadata_df.loc[
        values.between(
            selection.value - abs(selection.range),
            selection.value + abs(selection.range),
            inclusive="neither",
        ),
        [stage_cfg.metadata.dataframe_columns.fov_column_name, column],
    ]
    display(selected)